# F-Test: 3 Different Applications

An F-statistic creates a ratio to explore the potential differences between 2 or more groups. There are different ways to create this ratio based on what question you are trying to answer:
- Is there equal variance between 2 groups?
- Is there equal mean between 3 or more groups?
- Is this linear regression significant?

We will cover the third case in the Regression section of our course, so let's cover the first two here. 

## F-Distribution

The F-Distribution is a ratio of two chi-square distribution variables $U_1$ and $U_2$ with degrees of freedom $d_1$ and $d_2$:

$X = \frac{U_1/d_1}{U_2/d_2}$

![F-Distribution](images/f_distribution.png)

## F-Test for Equal Variance (2 Groups)

Two population Z-statistics and T-statistics attempt to discover if there is a difference in two populations' mean or proportion. If we instead want to explore if there is a difference in two populations' variance, we can construct an F-statistic by calculating the ratio of the two samples variances:

$$ F = \frac{s_1^2}{s_2^2} $$

Where $s_i^2$ is the sample variance for group $i$.

As this is a fairly simple statistic to construct, Python and Excel do not have a quick function to use. 

### Connection between formal definition and F-statistic for variance

You may be curious how we can go from independent variables of chi-squared distributions to just dividing variance. It is, surprise surprise, due to the chi-squared distribution being the sum of squares of independent standard normal random variables. This means we can write

$$ \frac{U_i}{d_i} = \frac{s^2_i}{\sigma^2_i} $$

where $s^2_i = S^2_i/d_i$ with $S^2_i$ as the sum of squares of $d_i$ random variables.

Let's plug this into our formula for F:

$$ F = \frac{s^2_1}{\sigma^2_1} \div \frac{s^2_2}{\sigma^2_2} = \frac{s^2_1 \cdot \sigma^2_2}{\sigma^2_1 \cdot s^2_2} $$

Our null hypothesis is that the variances are the same, so $\sigma^2_1 = \sigma^2_2$. So we can continue reducing:

$$ F = \frac{s^2_1 \cdot \sigma^2_2}{\sigma^2_1 \cdot s^2_2} = \frac{s^2_1}{s^2_2} \cdot \frac{\sigma^2_2}{\sigma^2_1} = \frac{s^2_1}{s^2_2} \cdot 1 = \frac{s^2_1}{s^2_2}$$

### Example F-Test for Equal Variance

In the "T-Test" lesson, we wanted to check if we could assume population variance for BMI was the same for people with and without diabetes. We did a quick calculation to show they **looked** similar enough to make that assumption, but let us statistically test this. 

#### Load Data

In [1]:
import pandas as pd
import os

diabetes_file_path = os.path.join(os.getcwd(), 'datasets', 'Diabetes_and_LifeStyle_Dataset.csv')
diabetes_df = pd.read_csv(diabetes_file_path)

diabetes_only_df = diabetes_df[diabetes_df['diagnosed_diabetes'] == 1].copy()
bmi_yes_diabetes_series = diabetes_only_df['bmi']

no_diabetes_df = diabetes_df[diabetes_df['diagnosed_diabetes'] == 0].copy()
bmi_no_diabetes_series = no_diabetes_df['bmi']

#### Hypotheses

$H_0: \sigma^2_n = \sigma^2_y$ <br>
$H_1: \sigma^2_n \neq \sigma^2_y$ 

Where $\sigma^2_n$ is the population variance for BMI without diabetes (no diabetes) and $\sigma^2_y$ is the population variance for BMI with diabetes (yes diabetes).

#### Test Statistic


In [ ]:
bmi_yes_var = bmi_yes_diabetes_series.var()
bmi_no_var = bmi_no_diabetes_series.var()

# Since the F-distribution is not symetric, we want to ensure we we use the upper tail for the cdf
# This occurs when F-statistic > 1, which is why we flip the fraction to ensure f_stat_bmi > 1
if bmi_yes_var > bmi_no_var:
    f_stat_bmi = bmi_yes_var / bmi_no_var
else:
    f_stat_bmi = bmi_no_var / bmi_yes_var

#### P-Value

In [10]:
from scipy.stats import f

df_yes = len(bmi_yes_diabetes_series)
df_no = len(bmi_no_diabetes_series)
p_value_bmi = 2*(1 - f.cdf(f_stat_bmi, df_yes, df_no))

In [11]:
print(p_value_bmi)

0.21265022908296638


#### Formal Result

Since p_value > 0.05, we fail to reject the null hypothesis that the population variance are the same between the two groups.

#### Meaning

Our sample variances suggest there is no significant difference between the populations' variance and so we CAN assume they are equal for purposes of the T-Test. 

## F-Test for Equal Mean

The F-Test for Equal Mean, also known as the one-way ANOVA, uses the variance of each group to determine if their means are significantly different from one another. It does this using sums of squares we are familiar with: it compares between-group variance and within-group variance. 

$$ F = \frac{\text{between-group variance}}{\text{within-group variance}} $$

While this is a little tougher to calculate since there are now 3 (or more) groups rather than just 2 as we saw before, both Excel and Python have functions to perform the calculation for you. 

### Example F-Test for Equal Mean

In the diabetes data, we have 5 different income levels: low, lower-middle, middle, upper-middle, and high. Let's do an F-test to see if there is a difference in the average BMI across all groups. 

#### Load Data

In [5]:
import pandas as pd
import os

diabetes_file_path = os.path.join(os.getcwd(), 'datasets', 'Diabetes_and_LifeStyle_Dataset.csv')
diabetes_df = pd.read_csv(diabetes_file_path)

group_low = diabetes_df[ diabetes_df['income_level'] == 'Low']['bmi']
group_lower_middle = diabetes_df[ diabetes_df['income_level'] == 'Lower-Middle']['bmi']
group_middle = diabetes_df[ diabetes_df['income_level'] == 'Middle']['bmi']
group_upper_middle = diabetes_df[ diabetes_df['income_level'] == 'Upper-Middle']['bmi']
group_high = diabetes_df[ diabetes_df['income_level'] == 'High']['bmi']

#### Hypotheses

$H_0: \mu_l = \mu_{lm} = \mu_{m} = \mu_{um} = \mu_{h} $ <br>
$H_0: \mu$ is different for at least one of the groups

#### Test Statistic and P-Value

In [6]:
from scipy.stats import f_oneway

f_stat_means, p_value_means = f_oneway(group_low, group_lower_middle, group_middle, group_upper_middle, group_high)

In [7]:
print(f'F-Statistic for Equal Means: {round(f_stat_means, 3)}')
print(f'P-Value for F-Statistic: {round(p_value_means, 3)}')

F-Statistic for Equal Means: 1.049
P-Value for F-Statistic: 0.38


#### Formal Result

Since p-value > 0.05, we fail to reject the null hypothesis that the mean BMI for all groups is the same. 

#### Meaning

Our sample suggests the average BMI is the same for all levels of income. 